# Cognifyz Technologies – Data Science Internship
## Level 2 Tasks
**Tasks:** Table Booking & Online Delivery | Price Range Analysis | Feature Engineering


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (10, 5)

df = pd.read_csv('Dataset_.csv', encoding='latin-1')
print(f"Dataset shape: {df.shape}")
df.head(3)


FileNotFoundError: [Errno 2] No such file or directory: 'Dataset_.csv'

---
## Task 1 – Table Booking and Online Delivery
1. Percentage of restaurants offering table booking and online delivery  
2. Compare average ratings with/without table booking  
3. Online delivery availability across price ranges


In [ ]:
# 1a. Percentage offering table booking & online delivery
tb_pct = (df['Has Table booking'] == 'Yes').mean() * 100
od_pct = (df['Has Online delivery'] == 'Yes').mean() * 100

print(f"Restaurants with Table Booking : {tb_pct:.2f}%")
print(f"Restaurants with Online Delivery: {od_pct:.2f}%")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, col, pct, title in zip(
    axes,
    ['Has Table booking', 'Has Online delivery'],
    [tb_pct, od_pct],
    ['Table Booking', 'Online Delivery']
):
    ax.pie([pct, 100-pct], labels=['Yes', 'No'],
           autopct='%1.1f%%', startangle=90,
           colors=['#4CAF50','#FF7043'])
    ax.set_title(f'Restaurants with {title}', fontsize=13, fontweight='bold')

plt.suptitle('Table Booking & Online Delivery Availability', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('l2_t1_pie.png', bbox_inches='tight', dpi=150)
plt.show()


In [ ]:
# 1b. Average ratings: with vs without table booking
rating_tb = df.groupby('Has Table booking')['Aggregate rating'].mean().reset_index()
rating_tb.columns = ['Has Table Booking', 'Average Rating']
print(rating_tb)

ax = sns.barplot(data=rating_tb, x='Has Table Booking', y='Average Rating',
                 palette=['#FF7043','#4CAF50'])
ax.set_title('Average Rating: With vs Without Table Booking', fontsize=13, fontweight='bold')
ax.set_ylim(0, 5)
for p in ax.patches:
    ax.annotate(f'{p.get_height():.2f}', (p.get_x()+p.get_width()/2, p.get_height()),
                ha='center', va='bottom', fontsize=12)
plt.tight_layout()
plt.savefig('l2_t1_rating_tb.png', dpi=150)
plt.show()


In [ ]:
# 1c. Online delivery availability by price range
od_price = df.groupby('Price range')['Has Online delivery'].apply(
    lambda x: (x=='Yes').mean()*100).reset_index()
od_price.columns = ['Price Range', 'Online Delivery %']
print(od_price)

ax = sns.barplot(data=od_price, x='Price Range', y='Online Delivery %', palette='Blues_d')
ax.set_title('Online Delivery Availability by Price Range', fontsize=13, fontweight='bold')
ax.set_ylabel('% Restaurants with Online Delivery')
for p in ax.patches:
    ax.annotate(f'{p.get_height():.1f}%', (p.get_x()+p.get_width()/2, p.get_height()),
                ha='center', va='bottom', fontsize=11)
plt.tight_layout()
plt.savefig('l2_t1_od_price.png', dpi=150)
plt.show()


---
## Task 2 – Price Range Analysis
1. Most common price range  
2. Average rating per price range  
3. Color representing the highest average rating


In [ ]:
# 2a. Most common price range
price_counts = df['Price range'].value_counts().reset_index()
price_counts.columns = ['Price Range', 'Count']
most_common = price_counts.iloc[0]['Price Range']
print(f"Most common price range: {most_common}")
print(price_counts)

ax = sns.barplot(data=price_counts, x='Price Range', y='Count', palette='Oranges_d')
ax.set_title('Distribution of Price Ranges', fontsize=13, fontweight='bold')
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x()+p.get_width()/2, p.get_height()),
                ha='center', va='bottom', fontsize=11)
plt.tight_layout()
plt.savefig('l2_t2_price_dist.png', dpi=150)
plt.show()


In [ ]:
# 2b. Average rating per price range
avg_rating_price = df.groupby('Price range')['Aggregate rating'].mean().reset_index()
avg_rating_price.columns = ['Price Range', 'Average Rating']
print(avg_rating_price)

ax = sns.barplot(data=avg_rating_price, x='Price Range', y='Average Rating', palette='YlOrRd')
ax.set_title('Average Rating by Price Range', fontsize=13, fontweight='bold')
ax.set_ylim(0, 5)
for p in ax.patches:
    ax.annotate(f'{p.get_height():.2f}', (p.get_x()+p.get_width()/2, p.get_height()),
                ha='center', va='bottom', fontsize=11)
plt.tight_layout()
plt.savefig('l2_t2_avg_rating_price.png', dpi=150)
plt.show()


In [ ]:
# 2c. Color representing highest average rating per price range
color_rating = df.groupby(['Price range','Rating color'])['Aggregate rating'].mean().reset_index()
best_color = color_rating.loc[color_rating['Aggregate rating'].idxmax()]
print(f"\nColor with highest average rating: {best_color['Rating color']}")
print(f"Price Range: {best_color['Price range']}, Avg Rating: {best_color['Aggregate rating']:.2f}")

pivot = color_rating.pivot(index='Price range', columns='Rating color', values='Aggregate rating').fillna(0)
pivot.plot(kind='bar', figsize=(12,5), colormap='tab10')
plt.title('Average Rating by Price Range and Rating Color', fontsize=13, fontweight='bold')
plt.ylabel('Average Rating')
plt.xlabel('Price Range')
plt.xticks(rotation=0)
plt.legend(title='Rating Color', bbox_to_anchor=(1.05,1))
plt.tight_layout()
plt.savefig('l2_t2_color_rating.png', dpi=150)
plt.show()


---
## Task 3 – Feature Engineering
1. Extract new features: name length, address length  
2. Encode categorical variables: Has Table Booking, Has Online Delivery


In [ ]:
# 3a. Extract length-based features
df['Restaurant Name Length'] = df['Restaurant Name'].astype(str).apply(len)
df['Address Length'] = df['Address'].astype(str).apply(len)

print("Sample of new features:")
print(df[['Restaurant Name','Restaurant Name Length','Address','Address Length']].head())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df['Restaurant Name Length'], bins=30, kde=True, ax=axes[0], color='steelblue')
axes[0].set_title('Distribution of Restaurant Name Length', fontweight='bold')

sns.histplot(df['Address Length'], bins=30, kde=True, ax=axes[1], color='coral')
axes[1].set_title('Distribution of Address Length', fontweight='bold')

plt.tight_layout()
plt.savefig('l2_t3_length_dist.png', dpi=150)
plt.show()


In [ ]:
# 3b. Encode categorical variables
df['Has Table Booking Encoded'] = (df['Has Table booking'] == 'Yes').astype(int)
df['Has Online Delivery Encoded'] = (df['Has Online delivery'] == 'Yes').astype(int)

print("Encoding Summary:")
print(df[['Has Table booking','Has Table Booking Encoded',
          'Has Online delivery','Has Online Delivery Encoded']].value_counts().reset_index())

# Correlation of new features with Aggregate rating
feature_cols = ['Restaurant Name Length','Address Length',
                'Has Table Booking Encoded','Has Online Delivery Encoded','Aggregate rating']
corr = df[feature_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Heatmap – Engineered Features vs Aggregate Rating', fontweight='bold')
plt.tight_layout()
plt.savefig('l2_t3_corr.png', dpi=150)
plt.show()

print("\nFeature Engineering complete. New columns added:")
print(['Restaurant Name Length','Address Length','Has Table Booking Encoded','Has Online Delivery Encoded'])


---
## ✅ Level 2 Complete